# Neo4j GraphRAG + Haystack

`neo4j-graphrag` is Neo4j's official retrieval library. It provides **retrievers** — objects that
turn a question into results from your graph using vector search, full-text search, graph
traversal, or a combination of all three.

This notebook connects those retrievers to a Haystack agent, so the agent picks a retrieval
strategy the way it would pick any other tool.

| | |
| --- | --- |
| **`neo4j-graphrag`** | Retrieval — finding the right context in the graph |
| **Haystack** | Orchestration — deciding what to retrieve, and writing the answer |

Contents:

1. Pairing an embedding model with a vector index
2. `VectorRetriever` — semantic search over text
3. `VectorCypherRetriever` — semantic search that continues into the graph
4. `HybridRetriever` / `HybridCypherRetriever` — semantic and keyword search combined
5. Retrievers as Haystack tools, chosen by an agent

Everything runs against the public Neo4j demo database. You need an OpenAI API key and nothing else.

---
## 1. Setup

In [1]:
%pip install -q "haystack-ai>=3.0" neo4j-graphrag neo4j sentence-transformers

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [ ]:
import os, json, logging, warnings
from getpass import getpass

import neo4j
from neo4j import GraphDatabase

from neo4j_graphrag.embeddings import SentenceTransformerEmbeddings
from neo4j_graphrag.retrievers import (
    VectorRetriever,
    VectorCypherRetriever,
    HybridRetriever,
    HybridCypherRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from haystack import component
from haystack.dataclasses import Document, ChatMessage
from haystack.tools import Tool
from haystack.components.agents import Agent
from haystack.components.generators.chat import OpenAIChatGenerator

warnings.filterwarnings("ignore")
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

print("ready")

In [1]:
os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

OPENAI_MODEL = "gpt-5.4-mini"

NEO4J_URI = "neo4j+s://demo.neo4jlabs.com"
NEO4J_DATABASE = "companies"
NEO4J_USERNAME = "companies"
NEO4J_PASSWORD = "companies"

NameError: name 'getpass' is not defined

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
print("Connected to", NEO4J_DATABASE)

### Cypher version

`neo4j-graphrag` generates its vector searches using the Cypher 25 `SEARCH` clause. The demo
database understands Cypher 25 but still defaults to Cypher 5, so we prefix the generated queries
to select the newer language version.

If your own database already defaults to Cypher 25 — set with
`ALTER DATABASE ... SET DEFAULT LANGUAGE CYPHER 25` — you can skip this cell.

In [ ]:
_execute_query = neo4j.Driver.execute_query

if not getattr(neo4j.Driver.execute_query, "_cypher25_patch", False):
    def _execute_query_cypher25(self, query, *args, **kwargs):
        if isinstance(query, str) and "VECTOR INDEX" in query.upper() \
                and not query.lstrip().upper().startswith("CYPHER"):
            query = "CYPHER 25 " + query
        return _execute_query(self, query, *args, **kwargs)

    _execute_query_cypher25._cypher25_patch = True
    neo4j.Driver.execute_query = _execute_query_cypher25

print("Generated vector queries will run as Cypher 25.")

---
## 2. The graph

The demo `companies` database holds news articles about public companies. Each article is split
into `Chunk` nodes that carry the text and its embeddings, and is linked to the organizations it
mentions:

```
(Article)-[:HAS_CHUNK]->(Chunk)
(Article)-[:MENTIONS]->(Organization)
(Organization)-[:HAS_COMPETITOR|HAS_INVESTOR|HAS_SUPPLIER]->(Organization)
```

That shape is what makes graph retrieval worthwhile. A vector search finds a *chunk* — but the
article it belongs to, its publication date, its sentiment score, and the companies it discusses
are all one hop away.

---
## 3. Pair the embedding model with the index

A vector index stores embeddings produced by one specific model. Querying it with a different
model still returns results, but they are meaningless: two models place the same text at
different points in space, so the similarity scores are noise.

Nothing raises an exception when this happens, which makes it the most common way a GraphRAG
setup silently underperforms. So start by looking at what indexes exist.

In [ ]:
indexes, _, _ = driver.execute_query(
    """
    SHOW INDEXES YIELD name, type, labelsOrTypes, properties, options
    WHERE type IN ['VECTOR', 'FULLTEXT'] AND labelsOrTypes = ['Chunk']
    RETURN name, type, properties,
           options.indexConfig['vector.dimensions'] AS dimensions
    ORDER BY type, name
    """,
    database_=NEO4J_DATABASE,
)

for record in indexes:
    row = record.data()
    dims = f"{row['dimensions']} dims" if row["dimensions"] else "-"
    print(f"  {row['name']:<18} {row['type']:<9} {str(row['properties']):<26} {dims}")

The demo graph stores several embeddings of the same chunks, one per model, so you can pick
whichever you have access to.

We use **`news_sbert`** with `SentenceTransformerEmbeddings`. Its 384 dimensions correspond to
`all-MiniLM-L6-v2`, the library's default sentence-transformers model, so the pairing is exact.
It also runs locally, which means no API key, no quota, and identical results every time you run
this notebook.

**`news_fulltext`** indexes the `text` property on those same `Chunk` nodes. Hybrid retrieval
needs a vector index and a full-text index over the same nodes, so this pairing is what makes
section 5 possible.

In [ ]:
VECTOR_INDEX = "news_sbert"
FULLTEXT_INDEX = "news_fulltext"

# Downloads all-MiniLM-L6-v2 on first use (about 90 MB).
embedder = SentenceTransformerEmbeddings()

print(f"{len(embedder.embed_query('test'))} dimensions")

---
## 4. `VectorRetriever` — semantic search

The simplest retriever. Give it a driver, an index name, and an embedder; it embeds the question,
searches the index, and returns matching nodes.

Set `return_properties` to control what comes back. Without it you get whole nodes — embeddings
included, which is thousands of floats per result heading straight into your prompt.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=NEO4J_DATABASE,
)

QUERY = "renewable energy investment"

for item in vector_retriever.search(query_text=QUERY, top_k=3).items:
    print(f"  {item.content[:160]}...\n")

This is ordinary RAG: it finds relevant text and stops. The retriever knows a chunk matched, but
not which article it came from, when it was published, or which companies it discusses.

---
## 5. `VectorCypherRetriever` — retrieval that continues into the graph

This is the retriever that separates GraphRAG from RAG.

It runs the same vector search, then executes a `retrieval_query` starting from each match. Two
variables are in scope for that query: `node`, the node found by the vector search, and `score`,
its similarity. From there you can traverse anywhere in the graph.

Below: chunk → article (title, date, sentiment) → organizations mentioned → their competitors.
The model receives the text *and* the surrounding facts in a single round trip.

A `result_formatter` turns each returned record into whatever shape you want. Without one you get
the driver's default record repr, which is awkward to read and awkward to hand to a model.

In [ ]:
RETRIEVAL_QUERY = """
WITH node AS chunk, score
MATCH (article:Article)-[:HAS_CHUNK]->(chunk)
OPTIONAL MATCH (article)-[:MENTIONS]->(org:Organization)
OPTIONAL MATCH (org)-[:HAS_COMPETITOR]->(rival:Organization)
RETURN chunk.text             AS text,
       article.id             AS article_id,
       article.title          AS title,
       toString(article.date) AS date,
       article.sentiment      AS sentiment,
       collect(DISTINCT org.name)[..5]   AS companies,
       collect(DISTINCT rival.name)[..5] AS competitors,
       score
ORDER BY score DESC
"""


def format_record(record) -> RetrieverResultItem:
    """Shape each result into compact JSON for the model."""
    return RetrieverResultItem(
        content=json.dumps({
            "text": (record.get("text") or "")[:600],
            "article_id": record.get("article_id"),
            "title": record.get("title"),
            "date": record.get("date"),
            "sentiment": record.get("sentiment"),
            "companies": record.get("companies"),
            "competitors": record.get("competitors"),
        }, default=str),
        metadata={"score": record.get("score")},
    )


graph_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=NEO4J_DATABASE,
)

for item in graph_retriever.search(query_text=QUERY, top_k=2).items:
    row = json.loads(item.content)
    print(f"  {row['title']}  ({row['date']}, sentiment {row['sentiment']})")
    print(f"    companies: {row['companies']}")
    print(f"    competitors: {row['competitors']}")
    print(f"    {row['text'][:120]}...\n")

Same query and same index as section 4, but every result now carries an article ID, a date, a
stored sentiment score, and the companies involved. That buys three things plain vector search
cannot offer:

* **Citations you can check.** `article_id` is a real node, so a claim can be traced back to its
  source and verified.
* **Facts the text never states.** A competitor relationship lives in the graph, not in the
  article body.
* **Structure to filter and rank on.** Date, sentiment, and relationships are all available after
  the semantic match.

---
## 6. `HybridRetriever` — semantic plus keyword

Vector search is strong on meaning and weak on exact strings. Ask about a specific company,
ticker, or product and it tends to return material on the right *topic* while missing the
document that names it outright.

Full-text search has the opposite profile: exact terms, no understanding of meaning.

`HybridRetriever` runs both and merges the rankings, which recovers the exact matches without
losing the semantic ones.

In [ ]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=["text"],
    neo4j_database=NEO4J_DATABASE,
)

NAMED_QUERY = "Nvidia data center GPU demand"

print("--- vector only ---")
for item in vector_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

print("\n--- hybrid ---")
for item in hybrid_retriever.search(query_text=NAMED_QUERY, top_k=3).items:
    print(f"  {item.content[:120]}...")

The hybrid results surface chunks naming the company directly, while vector-only results drift
toward the general theme. The more distinctive the term — proper nouns, tickers, product codes —
the wider the gap.

`HybridCypherRetriever` combines both ideas: hybrid search plus a `retrieval_query`. It is the
one to reach for in practice, and the one we give the agent below.

In [ ]:
hybrid_graph_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
    neo4j_database=NEO4J_DATABASE,
)

row = json.loads(hybrid_graph_retriever.search(query_text=NAMED_QUERY, top_k=1).items[0].content)
print(json.dumps(row, indent=2)[:600])

---
## 7. Retrievers as Haystack components

A retriever is a `neo4j-graphrag` object with a `search()` method. To use it inside a Haystack
`Pipeline` (as opposed to handing it to an agent as a tool, in section 8), wrap it once in a small
`@component` adapter that turns `RetrieverResultItem`s into Haystack `Document`s.

In [ ]:
@component
class GraphRAGRetrieverComponent:
    """Adapts any neo4j-graphrag retriever to the Haystack component interface."""

    def __init__(self, retriever, top_k: int = 5):
        self._retriever = retriever
        self._top_k = top_k

    @component.output_types(documents=list)
    def run(self, query: str, top_k: int = None):
        result = self._retriever.search(query_text=query, top_k=top_k or self._top_k)
        docs = []
        for item in result.items:
            try:
                row = json.loads(item.content)
                docs.append(Document(content=row.get("text", ""), meta=row))
            except (json.JSONDecodeError, TypeError):
                docs.append(Document(content=str(item.content)))
        return {"documents": docs}


vector_component = GraphRAGRetrieverComponent(vector_retriever)
graph_component = GraphRAGRetrieverComponent(graph_retriever)
hybrid_component = GraphRAGRetrieverComponent(hybrid_retriever)
hybrid_graph_component = GraphRAGRetrieverComponent(hybrid_graph_retriever)

print("4 retrievers wrapped as Haystack components.")

---
## 8. Retrievers as Haystack tools, chosen by an agent

A retriever is a Python object with a `search()` method, so exposing one to an agent means
wrapping it in a function. Haystack's `Tool` builds the tool schema from the name, description,
and JSON parameter schema you give it — so **the description is the interface**. It is how the
model decides which retriever fits the question in front of it.

Below, three retrieval strategies with genuinely different strengths, and an agent that chooses.

In [ ]:
def search_news(question: str) -> str:
    """Search news article text by meaning. Best for broad themes and open-ended questions,
    for example "what are the concerns around AI regulation"."""
    result = vector_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


def search_news_with_context(question: str) -> str:
    """Search news by meaning and return graph context with each result: the source article,
    its date and sentiment, the companies mentioned, and their competitors. Use this whenever
    the answer needs citations or company relationships."""
    result = hybrid_graph_retriever.search(query_text=question, top_k=5)
    return json.dumps([json.loads(item.content) for item in result.items], indent=2)


def find_named_entity(question: str) -> str:
    """Search news for an exact company name, ticker, or product that appears literally in the
    text. Use this when the question centres on a specific named thing rather than a theme."""
    result = hybrid_retriever.search(query_text=question, top_k=5)
    return json.dumps([item.content for item in result.items], indent=2)


def _tool(fn, description):
    return Tool(
        name=fn.__name__,
        description=description,
        function=fn,
        parameters={
            "type": "object",
            "properties": {"question": {"type": "string", "description": "Natural-language query"}},
            "required": ["question"],
        },
    )


tools = [
    _tool(search_news, search_news.__doc__),
    _tool(search_news_with_context, search_news_with_context.__doc__),
    _tool(find_named_entity, find_named_entity.__doc__),
]

news_analyst = Agent(
    chat_generator=OpenAIChatGenerator(model=OPENAI_MODEL),
    tools=tools,
    system_prompt="""You answer questions about companies using a Neo4j news graph.

Choose the retrieval tool that fits the question:
- `search_news` for broad themes and open-ended topics.
- `search_news_with_context` when the answer needs citations, dates, sentiment, or company
  relationships. Prefer this one for anything analytical.
- `find_named_entity` when the question is about a specific named company, ticker, or product.

Cite the article title and date when the tool provides them. If the retrieved passages do not
answer the question, say so rather than filling the gap from your own knowledge.
""",
)

print("Agent ready with 3 retrieval strategies.")

In [ ]:
def ask(agent, query, show_tools=True):
    print(f"USER: {query}")
    result = agent.run(messages=[ChatMessage.from_user(query)])

    for msg in result["messages"]:
        if show_tools and msg.tool_call:
            print(f"   -> {msg.tool_call.tool_name}({msg.tool_call.arguments})")
        elif show_tools and msg.tool_call_result:
            print(f"   <- {msg.tool_call_result.origin.tool_name}")

    answer = result["messages"][-1].text
    print(f"\nAGENT: {answer}\n")
    return answer

In [ ]:
# Analytical question - needs relationships and citations.
ask(news_analyst, "Which companies are expanding in renewable energy, and who competes with them?")

In [ ]:
# Broad thematic question - semantic search alone is enough.
ask(news_analyst, "What are the recent trends in cloud computing?")

---
## 9. Comparing strategies side by side

Tool choice is the agent's problem. Knowing which retriever suits *your* data is yours. Run all
three over one question and read the results together.

In [ ]:
COMPARISON_QUERY = "semiconductor manufacturing capacity"

for label, retriever in [
    ("vector", vector_retriever),
    ("hybrid", hybrid_retriever),
    ("hybrid + graph", hybrid_graph_retriever),
]:
    print(f"\n=== {label} ===")
    for item in retriever.search(query_text=COMPARISON_QUERY, top_k=2).items:
        try:
            row = json.loads(item.content)
            print(f"  [{row['title']}] {row['text'][:130].strip()}...")
        except (json.JSONDecodeError, KeyError):
            print(f"  {item.content[:150].strip()}...")

Judge them on whether the retrieved passages would let a model answer your questions: recall
across the topic, precision on named entities, and whether the extra graph context justifies the
extra Cypher.

---
## 10. Cleanup

In [ ]:
driver.close()
print("Closed.")

---
## Summary

**Retrievers are the unit of reuse.** Each one bundles an index, an embedding model, and
optionally a traversal query behind a single `search()` method. Swapping `VectorRetriever` for
`HybridCypherRetriever` changes retrieval behaviour without touching the agent or pipeline
around it.

**Pair the embedding model with the index deliberately.** A mismatch does not raise an error, it
just returns quietly useless results. Check dimensions, and re-embed a stored chunk to confirm the
models agree.

**`VectorCypherRetriever` is where the graph earns its place.** Vector search finds a chunk; the
`retrieval_query` walks outward to articles, organizations, and relationships. The model receives
text plus checkable structure in one round trip.

**Hybrid search covers what embeddings miss.** Semantic similarity is weak on proper nouns and
codes. Running vector and full-text together costs little and recovers the exact matches.

**Haystack turns retrieval strategies into a decision.** Wrapped as `Tool`s, several retrievers
become options an `Agent` selects between per question — and their descriptions are the entire
control surface. The same wrapped retrievers also drop straight into a `Pipeline` (section 7) when
you want to fix the strategy instead of letting the agent choose.

### Next steps

* [GraphRAG user guide](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html) —
  `Text2CypherRetriever`, `ToolsRetriever`, and the `GraphRAG` generation pipeline
* [Knowledge graph builder](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html) —
  `SimpleKGPipeline` for building a graph from unstructured documents
* [API reference](https://neo4j.com/docs/neo4j-graphrag-python/current/api.html) — every retriever,
  embedder, and LLM interface
* [Haystack Agents](https://docs.haystack.deepset.ai/docs/agents) — fixing the retrieval strategy
  instead of letting the agent choose